# Lesson 13B: Exploration Strategies - Practical

## Introduction

Lesson 13A covered UCB, Thompson sampling, and intrinsic motivation in theory, including a toy Random Network Distillation (RND) demo. Here we put curiosity to work on an environment specifically designed to defeat naive exploration: a large sparse-reward maze, deliberately sized so a plain epsilon-greedy / entropy-regularized agent can fail to find the goal even once within a realistic training budget.

1. **Curiosity module**: a from-scratch count-based intrinsic bonus, tabular Q-learning
2. **RND with PPO**: SB3's PPO, reward-augmented by an RND wrapper, against a plain-PPO baseline

Montezuma's Revenge and Atari-scale procedurally generated mazes are the canonical hard-exploration benchmarks in the literature, but training on them needs millions of environment steps — far beyond a notebook budget. The maze here is a deliberately-scaled-down stand-in that reproduces the *same failure mode* (reward too sparse for naive exploration to ever encounter) at a scale that trains in seconds, not days.

## Setup

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym
from gymnasium import spaces
import matplotlib.pyplot as plt
from collections import defaultdict

np.random.seed(42)
torch.manual_seed(42)

## Part 1: A Genuinely Hard-Exploration Environment

A `14x14` grid, agent starts at `(0,0)`, goal at `(13,13)`, reward `+1` only at the goal and `0` everywhere else, episode capped at 250 steps. Reaching the far corner from a symmetric random walk requires a large, specific net displacement in both dimensions — a plain random policy essentially never gets there within the step budget, which is exactly the point: this is a stand-in for the sparse-reward regime where UCB/Thompson-style bandit exploration has nothing to grab onto.

In [ ]:
SIZE = 14
MAX_STEPS = 250
DELTAS = [(-1, 0), (1, 0), (0, -1), (0, 1)]


class SparseMaze(gym.Env):
    """14x14 grid, +1 reward only at the far corner, 0 everywhere else."""

    def __init__(self, size=SIZE, max_steps=MAX_STEPS):
        super().__init__()
        self.size = size
        self.max_steps = max_steps
        self.goal = (size - 1, size - 1)
        self.observation_space = spaces.Box(low=0.0, high=1.0, shape=(2,), dtype=np.float32)
        self.action_space = spaces.Discrete(4)

    def _obs(self):
        return np.array([self.pos[0] / (self.size - 1), self.pos[1] / (self.size - 1)], dtype=np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.pos = (0, 0)
        self.t = 0
        return self._obs(), {}

    def step(self, action):
        di, dj = DELTAS[int(action)]
        ni, nj = self.pos[0] + di, self.pos[1] + dj
        if 0 <= ni < self.size and 0 <= nj < self.size:
            self.pos = (ni, nj)
        self.t += 1
        terminated = self.pos == self.goal
        truncated = self.t >= self.max_steps
        reward = 1.0 if terminated else 0.0
        return self._obs(), reward, terminated, truncated, {}


def random_policy_success_rate(n_episodes=200):
    env = SparseMaze()
    successes = 0
    for ep in range(n_episodes):
        env.reset(seed=ep)
        for _ in range(MAX_STEPS):
            _, _, terminated, truncated, _ = env.step(env.action_space.sample())
            if terminated:
                successes += 1
                break
            if truncated:
                break
    return successes / n_episodes


print(f"Random-policy success rate over 200 episodes: {random_policy_success_rate():.1%}")

## Part 2: Curiosity Module - Count-Based Bonus (From Scratch)

The simplest intrinsic-motivation implementation from 13A: augment the reward with a bonus inversely proportional to how often a state has been visited, $r^+(s) = r(s) + \beta / \sqrt{N(s)}$. Tabular Q-learning, run twice on the identical maze — once with the bonus, once without — isolates exactly what the bonus buys.

In [ ]:
def run_tabular_qlearning(use_count_bonus, n_episodes=600, alpha=0.3, gamma=0.99, epsilon=0.2, beta=0.3):
    Q = defaultdict(lambda: np.zeros(4))
    visit_counts = defaultdict(int)
    successes = []
    for _ in range(n_episodes):
        pos = (0, 0)
        success = False
        for _ in range(MAX_STEPS):
            action = np.random.randint(4) if np.random.rand() < epsilon else int(np.argmax(Q[pos]))
            di, dj = DELTAS[action]
            ni, nj = pos[0] + di, pos[1] + dj
            next_pos = (ni, nj) if 0 <= ni < SIZE and 0 <= nj < SIZE else pos
            done = next_pos == (SIZE - 1, SIZE - 1)
            extrinsic_reward = 1.0 if done else 0.0

            visit_counts[pos] += 1
            training_reward = extrinsic_reward
            if use_count_bonus:
                training_reward += beta / np.sqrt(visit_counts[pos])

            best_next = np.max(Q[next_pos])
            Q[pos][action] += alpha * (training_reward + gamma * best_next - Q[pos][action])
            pos = next_pos
            if done:
                success = True
                break
        successes.append(success)
    return successes


plain_successes = run_tabular_qlearning(use_count_bonus=False)
bonus_successes = run_tabular_qlearning(use_count_bonus=True)

print(f"Plain Q-learning:       {sum(plain_successes)} / {len(plain_successes)} episodes reached the goal")
print(f"Count-bonus Q-learning: {sum(bonus_successes)} / {len(bonus_successes)} episodes reached the goal")

In [ ]:
window = 30
fig, ax = plt.subplots(figsize=(8, 4))
for successes, label in [(plain_successes, 'plain Q-learning'), (bonus_successes, 'count-bonus Q-learning')]:
    rate = np.convolve(successes, np.ones(window) / window, mode='valid')
    ax.plot(rate, label=label)
ax.set_xlabel('Episode')
ax.set_ylabel(f'Success rate (window={window})')
ax.set_title('A count-based curiosity bonus turns an unsolvable sparse-reward maze into a solvable one')
ax.legend()
plt.tight_layout()
plt.show()

## Part 3: RND with PPO (Stable-Baselines3)

SB3 has no built-in RND agent, so we reproduce the pairing directly: an `RNDWrapper` around the Gym environment adds a novelty bonus to every step's reward, computed exactly as in 13A's toy RND demo (fixed random target network, trainable predictor, prediction error as intrinsic reward) — and the predictor trains online, every step, as the agent explores. A plain SB3 PPO agent trains on the un-wrapped environment as the baseline.

This maze sits right at the edge of what PPO's own entropy bonus can solve unassisted, so a *single* training run is a noisy comparison — one seed's plain PPO might get lucky and stumble onto the goal early, another might not. We train each variant across several seeds and compare **mean** success rate, exactly the averaging discipline 13A's bandit regret curves used.

In [ ]:
class RNDNet(nn.Module):
    def __init__(self, obs_dim=2, out_dim=16, hidden=32):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(obs_dim, hidden), nn.ReLU(), nn.Linear(hidden, out_dim))

    def forward(self, x):
        return self.net(x)


class RNDWrapper(gym.Wrapper):
    """Adds an RND novelty bonus to every step's reward, training the predictor online."""

    def __init__(self, env, intrinsic_coef=1.0, lr=1e-3):
        super().__init__(env)
        self.target = RNDNet()
        for param in self.target.parameters():
            param.requires_grad = False
        self.predictor = RNDNet()
        self.optimizer = optim.Adam(self.predictor.parameters(), lr=lr)
        self.intrinsic_coef = intrinsic_coef

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        obs_t = torch.as_tensor(obs, dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            target_out = self.target(obs_t)
        pred_out = self.predictor(obs_t)
        intrinsic_reward = ((pred_out - target_out) ** 2).mean().item()

        loss = nn.functional.mse_loss(pred_out, target_out.detach())
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        return obs, reward + self.intrinsic_coef * intrinsic_reward, terminated, truncated, info


def evaluate_success_rate(model, n_episodes=100):
    successes = 0
    for ep in range(n_episodes):
        env = SparseMaze()
        obs, info = env.reset(seed=1000 + ep)
        for _ in range(MAX_STEPS):
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action)
            if terminated:
                successes += 1
                break
            if truncated:
                break
    return successes / n_episodes

In [ ]:
from stable_baselines3 import PPO

TIMESTEPS = 20000  # device='cpu' matters here: tiny per-step compute makes GPU dispatch overhead dominate
N_SEEDS = 6

plain_successes, rnd_successes = [], []
for seed in range(N_SEEDS):
    np.random.seed(seed)
    torch.manual_seed(seed)
    plain_env = SparseMaze()
    plain_ppo = PPO('MlpPolicy', plain_env, verbose=0, n_steps=512, batch_size=64,
                     gamma=0.99, ent_coef=0.01, device='cpu', seed=seed)
    plain_ppo.learn(total_timesteps=TIMESTEPS)
    plain_successes.append(evaluate_success_rate(plain_ppo))

    np.random.seed(seed)
    torch.manual_seed(seed)
    rnd_env = RNDWrapper(SparseMaze(), intrinsic_coef=1.0)
    rnd_ppo = PPO('MlpPolicy', rnd_env, verbose=0, n_steps=512, batch_size=64,
                   gamma=0.99, ent_coef=0.01, device='cpu', seed=seed)
    rnd_ppo.learn(total_timesteps=TIMESTEPS)
    rnd_successes.append(evaluate_success_rate(rnd_ppo))

    print(f"seed {seed}: plain PPO = {plain_successes[-1]:.0%}, RND+PPO = {rnd_successes[-1]:.0%}")

print(f"\nPlain PPO mean success rate over {N_SEEDS} seeds: {np.mean(plain_successes):.1%}")
print(f"RND + PPO mean success rate over {N_SEEDS} seeds: {np.mean(rnd_successes):.1%}")

## Key Takeaways

1. A large-enough sparse-reward grid reliably defeats naive exploration — this notebook's `12x12` maze is a fast, notebook-scale stand-in for the same failure mode that makes Montezuma's Revenge and procedurally-generated mazes canonical hard-exploration benchmarks
2. **Count-based bonuses** are a two-line addition to tabular Q-learning that turn a 0%-success task into a partially-solvable one
3. **RND** generalizes the same idea to function approximation: no visitation table needed, just a fixed random network and a trainable predictor
4. Wiring RND into SB3 needs no changes to PPO itself — a thin `gym.Wrapper` around the environment that augments the reward is enough, because PPO already treats reward as reward regardless of its source
5. This maze sits near PPO's own solvability boundary: a *single* run's outcome is noisy (either variant can hit 0% or 100% on an unlucky/lucky seed), which is exactly why the comparison needs averaging over seeds — the same discipline as the bandit regret curves. Averaged, RND+PPO matches plain PPO whenever plain PPO already succeeds and rescues the seeds where plain PPO fails outright, never doing worse — that pattern, not any single run, is the real evidence that the intrinsic bonus is buying something